In [19]:
from pathlib import Path
import os


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start


def load_env_file() -> None:
    repo_root = _find_repo_root(Path.cwd())
    env_path = repo_root / ".env"
    example_path = repo_root / ".env.example"
    target = env_path if env_path.exists() else example_path
    if not target.exists():
        raise FileNotFoundError(
            f"Expected either {env_path} or {example_path} to exist."
        )

    with target.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key, value)

    print(f"Loaded environment variables from {target.relative_to(repo_root)}")


load_env_file()


Loaded environment variables from .env


In [20]:
from langchain.agents import create_agent

In [21]:
def get_weather(city: str) -> str:
    """Get weather for a given city"""
    return f"Its always sunny in {city}"

agent = create_agent(
    model = 'openai:gpt-4o-mini',
    tools = [get_weather],
    prompt = 'You are a helpful assistant',
)

In [22]:
agent.invoke(
    {"message": [{"role":"user", "content": "what is the weather in sf"}]}
)

{'messages': [AIMessage(content='How can I assist you today? If you need information or have a question, feel free to ask!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 46, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CNd7KONPRNSCP6iB5vFL0rFrt2zHi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--d27435fa-5e29-41c8-bdbd-a8c7edef2917-0', usage_metadata={'input_tokens': 46, 'output_tokens': 22, 'total_tokens': 68, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

## Weather Forecasting Agent

In [9]:
# Step 1: System Prompt - The Agent's initial instructions or personality.
system_prompt = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean whereever they are, use the get_user_location tool to find their location."""

In [4]:
# Step 2: Create tools - tools are functions that can be called, they interact with external data to get stuff done.
from langchain_core.tools import tool
import random

def get_weather_for_location(city: str) -> str:
    '''Get weather for a given city'''
    conditions = random.choice(['sunny', 'rainy', 'cloudy'])
    return f'It is {conditions} in {city}'

from langchain_core.runnables import RunnableConfig

# A lookup table for demo purposes
USER_LOCATION = {
    "1":"Florida",
    "2":"SF"
}

'''
@tool decorator turns Python callables into LangChain `StructuredTool` objects
that the agent can discover and invoke. It can then use LangChain's tool metadata 
like names, descriptions, config injections.
'''
@tool
def get_user_location(config: RunnableConfig) -> str:
    '''Retrieve user information'''
    user_id = config.get("configurable", {}).get("user_id")
    return USER_LOCATION[user_id]


In [5]:
# Step 3: Configure the model
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0,
)

In [6]:
# Step 4: Define response format
from dataclasses import dataclass

@dataclass
class WeatherResponse:
    conditions: str
    punny_response: str

In [7]:
# Step 5: Add memory for the agent to remember conversation history
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [10]:
# Step 6: Bring it all together
agent = create_agent(
    model=model,
    prompt=system_prompt,
    tools=[get_user_location, get_weather_for_location],
    response_format=WeatherResponse,
    checkpointer=checkpointer
)

# config = {"configurable": {"thread_id": "1"}}
# context = {"user_id": "1"}

'''
`config` is the run metadata shared across every runnable (models, tools, graphs).
`config` has reserved keys like "configurable", "run_name", "tags", "metadata", "callbacks".
"configurable" is a catch-all for values we want to read back inside the graph or tools.
"thread_id" must be supplied when using `InMemorySaver` or any checkpointer - it decides which conversation thread to load.
'''

config = {"configurable": {"thread_id": "1", "user_id": "2"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
)

response['structured_response']

WeatherResponse(conditions='rainy', punny_response="Looks like it's a wet and wild day in SF! Don't forget your umbrella, or you might just get caught in a drizzle of disappointment!")

**More control over the model using provider's package**

```python
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
)
agent = create_agent(model, tools=tools)
```

In [29]:

response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config
)

response['structured_response']

WeatherResponse(conditions='sunny', punny_response="You're welcome! I'm always here to brighten your day!")

# More About Agents

In [12]:
# Dynamic Model Loading
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langgraph.runtime import Runtime

def select_model(state: AgentState, runtime: Runtime) -> ChatOpenAI:
    '''Chooses model based on conversation complexity'''
    messages = state['messages']
    message_count = len(messages)
    print(messages)

    if message_count < 10:
        return ChatOpenAI(model='gpt-4.1-mini').bind_tools(tools)
    else:
        return ChatOpenAI(model='gpt-5').bind_tools(tools)
    
tools = [get_user_location, get_weather_for_location]

# Pass this function as the model in `create_agent()`
agent = create_agent(select_model, tools=tools)
agent.invoke({"messages":[{"role": "user", "content": "what is the weather outside?"}]},
             config=config)

[HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='c5cd632e-fc64-481a-8d6a-2ce21164225f')]
[HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='c5cd632e-fc64-481a-8d6a-2ce21164225f'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_LWGamRPUdow7AhxPJzBLKJhP', 'function': {'arguments': '{}', 'name': 'get_user_location'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 65, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_95d112f245', 'id': 'chatcmpl-CNdY91dJyQXoPYSKQK3ne3k9braGC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='ru

{'messages': [HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='c5cd632e-fc64-481a-8d6a-2ce21164225f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_LWGamRPUdow7AhxPJzBLKJhP', 'function': {'arguments': '{}', 'name': 'get_user_location'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 65, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_95d112f245', 'id': 'chatcmpl-CNdY91dJyQXoPYSKQK3ne3k9braGC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--22c9bbe7-b98a-4cb5-986d-9b4e9bd8cda7-0', tool_calls=[{'name': 'get_user_location', 'args': {}, 'id': 'call_LWGamRPUdow7AhxPJ

In [ ]:
# ToolNode: Creating tools like before (callables, @tool, provider's dict) will internally create
# a ToolNode. Here, you can defined ToolNode yourself for finer control.
from langchain_core.tools import tool
from langchain.agents import ToolNode
from langchain.agents import create_agent

tool_node = ToolNode(
    tools = [get_user_location, get_weather_for_location],
    handle_tool_errors = 'check again if error'
)

'''
This should not have run without passing `config` but since we handled errors
using `handle_tool_errors`, it bypasses the error.
'''
agent = create_agent(model, tools=tool_node)
result = agent.invoke({"messages":[{"role": "user", "content": "what is the weather outside?"}]})
result

{'messages': [HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='3b8f39e2-3ebf-445a-b11c-3e2ddcf69896'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_vMjkgd9MCGdI5EZBg7jjIMwP', 'function': {'arguments': '{}', 'name': 'get_user_location'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 65, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CNd86iY2bobwHhscsoAS17E21Ct96', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--e96f59a1-2885-4a90-8d4c-6e7106df4ed9-0', tool_calls=[{'name': 'get_user_location', 'args': {}, 'id': 'call_vMjkgd9MCGdI5EZBg7

In [33]:
model = ChatOpenAI(
    model='gpt-4.1-mini'
)

* `langchain.agents.AgentState`: TypedDict used to track what agents knows so far (conversation messages, intermediate tool steps, next node to execute).
* `langgraph.prebuilt.tool_executor.ModelRequest`: a dataclass LangGraph emits whenever the agents needs model call. Carries prompt messages plus runtime config.
* `langgraph.runtime.Runtime`: event scheduler that drivers a LangGraph.

In [ ]:
# Dynamic Prompt with Middleware
from typing import TypedDict

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware.types import ModelRequest, modify_model_request
from langgraph.runtime import Runtime

class Context(TypedDict):
    user_role: str
'''
Can't use config = {"context":"..."} here because Middlewares registered with
@modify_model_request can only be populated by context= argument in agent.invoke;

Use `config` dict when working with plain LangChain runnables, tools, model wrappers etc.
These will pull values from RunnableConfig. Place custom values under the `configurable` key.

Use `context=` argument in agent.invoke when using LangGraph's run-scoped state
like middlewares, graph nodes that accept runtime, store helpers etc. They will
look at runtime.context.
'''


@modify_model_request
def dynamic_system_prompt(state: AgentState, request: ModelRequest, runtime: Runtime[Context]) -> ModelRequest:
    print(runtime.context)
    user_role = runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        prompt = f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        prompt = f"{base_prompt} Explain concepts simply and avoid jargon."
    else:
        prompt = base_prompt
    print(request)
    request['system_prompt'] = prompt
    # Can't do `request.system_prompt=` here because the request dict might not 
    # have the system_prompt key yet.
    return request

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=tools,
    middleware=[dynamic_system_prompt],
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    # {"context": {"user_role": "expert"}}
    context=Context(user_role='expert')
)

{'user_role': 'expert'}
{'messages': [HumanMessage(content='Explain machine learning', additional_kwargs={}, response_metadata={}, id='dbf83858-4296-4c49-8ddc-313a9ac548f5')]}


In [31]:
print(result['messages'][1].content)

Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. Instead of being programmed to perform specific tasks, machine learning systems learn from data and improve their performance over time.

Here are some key concepts and components of machine learning:

1. **Data**: Machine learning relies heavily on data, which can be in various forms such as text, images, audio, or numerical values. This data is used to train machine learning models.

2. **Training**: During the training phase, a machine learning model is exposed to a dataset that includes input-output pairs. The model learns to recognize patterns in the data, adjusting its internal parameters to minimize errors in predictions.

3. **Models**: A model is a mathematical representation of the relationship between input features and output predictions. Different types of models can be used d

In [34]:
from typing import TypedDict
from typing_extensions import Annotated
from langgraph.graph.message import add_messages
from langchain.agents import create_agent
from langchain.agents import AgentState

class CustomAgentState(AgentState):
    messages: Annotated[list, add_messages]
    user_preferences: dict

agent = create_agent(
    model,
    tools=tools,
    state_schema=CustomAgentState
)

# The agent can now track additional state beyond messages. This custom state can be accessed and updated throughout the conversation.
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "OneWord"},
})

In [36]:
result

{'messages': [HumanMessage(content='I prefer technical explanations', additional_kwargs={}, response_metadata={}, id='963b1d48-8bb1-4469-b9b0-c25e22db5266'),
  AIMessage(content="Sure! Please specify the topic or concept you'd like a technical explanation about, and I'll provide a detailed response.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 63, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CNegPeQQ8KeN1cQ4CuJrgHOGlImLJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--8e03bed1-d73b-47a8-babd-aaf312e06de0-0', usage_metadata={'input_tokens': 63, 'output_tokens': 23, 'total_tokens': 86, 'input_token_details': {'aud